# 02 · Embeddings & Cosine Similarity
Turn text into vectors, then measure closeness with cosine similarity — the mechanism behind semantic search.

In [ ]:
# A REAL but simple vectorizer: bag-of-words (term frequency). Offline, deterministic,
# and good enough that similar documents (sharing words) get similar vectors — so
# cosine similarity and retrieval metrics are genuinely meaningful and hand-checkable.
#
# NOTE: production RAG uses NEURAL embeddings (e.g. Jina, OpenAI, sentence-transformers)
# that also capture SYNONYMS ("car" ~ "automobile") with no shared words. The MATH below
# (cosine, retrieval, metrics) is identical; only the vectors get smarter. Where a cell
# says "swap in a real embedder", that's the one line that changes.
import numpy as np, re
def tokenize(text): return re.findall(r"[a-z0-9]+", text.lower())
def build_vocab(texts):
    vocab={}
    for t in texts:
        for w in tokenize(t):
            if w not in vocab: vocab[w]=len(vocab)
    return vocab
def vectorize(text, vocab):
    v=np.zeros(len(vocab))
    for w in tokenize(text):
        if w in vocab: v[vocab[w]]+=1.0
    return v
def cosine(a,b):
    na,nb=np.linalg.norm(a),np.linalg.norm(b)
    return float(a@b/(na*nb)) if na and nb else 0.0

## 1. Vectorize a few sentences

In [ ]:
sentences = [
    "how do I get a refund",
    "I want my money back",         # same meaning, different words
    "what credit cards do you take", # different topic
]
vocab = build_vocab(sentences)
vecs = [vectorize(s, vocab) for s in sentences]
print("vocabulary size:", len(vocab))
print("vector for sentence 0:", vecs[0].astype(int))

## 2. Cosine similarity: 1.0 = identical direction, 0 = unrelated

In [ ]:
print("sim(refund , money back) =", round(cosine(vecs[0], vecs[1]), 3))
print("sim(refund , credit cards) =", round(cosine(vecs[0], vecs[2]), 3))

**Observe (and an honest caveat):** with a bag-of-words vectorizer, 'get a refund' and 'want my money back' share almost NO words, so their similarity is low — even though they mean the same thing. This is exactly the limitation neural embeddings fix: a real embedding model places 'refund' and 'money back' close together *because it learned meaning*. The cosine math is identical; the vectors are smarter.

## 3. See it: cosine measures the ANGLE between vectors
```
   ^                 v1 and v2 point almost the SAME way  -> cosine near 1
   |   v1  v2         v1 and v3 point in DIFFERENT ways    -> cosine near 0
   |  /  /
   | /  /      v3
   |/  /  _____---->
   +--------------->
```
Cosine ignores length (how many words) and looks only at direction (which words) — that's why it works for documents of different sizes.

In [ ]:
# proof it ignores magnitude: double one vector -> same cosine
import numpy as np
print("cosine(v, v*5) =", round(cosine(vecs[0], vecs[0]*5), 3), "(scaling doesn't change direction)")